# Brain Tumor MRI — Colab (one cell)

**Runtime → Change runtime type → GPU** (recommended).

Open the code cell below, edit settings, then run that single cell. It mounts Drive, clones the repo, installs dependencies, merges your hyperparameters into **`configs/config_colab_runtime.yaml`** (regenerated every run), prepares data, trains, and prints **final_results.csv** at the end.

### Hyperparameters

In the code cell, adjust **`EPOCHS`**, **`MODELS`**, and the **Hyperparameters** block (`LR`, `BATCH_SIZE`, splits, etc.). Values override **configs/config_colab.yaml** in the repo before training. **`main.py`** is run with **`--config configs/config_colab_runtime.yaml`**.

### Google Drive dataset

1. Set **`DATA_SOURCE = "drive"`**.
2. Set **`DRIVE_DATA_PATH`** to the folder that contains **`Training/`** (and **`Testing/`** if present): `/content/drive/MyDrive/...` or `/content/drive/Shareddrives/<Name>/...`.
3. **`DRIVE_USE_IN_PLACE`**: `False` copies into `data/raw/` (often faster). `True` uses **`--data_dir`** on Drive (saves disk; can be slower).

### Kaggle dataset

Set **`DATA_SOURCE = "kaggle"`**. Put **`kaggle.json`** on Drive at **`KAGGLE_JSON_DRIVE`**, or upload when prompted.


In [ ]:
# =============================================================================
# EDIT THESE — then run this cell (Runtime → Change runtime type → GPU recommended)
# =============================================================================
EPOCHS = 2
# One model or several (space-separated): resnet50 | vit | hybrid
MODELS = "resnet50"

REPO_URL = "https://github.com/MuhammadSaljooq/Tumor-AI-training-model.git"
BRANCH = "checkpoint_added"
PROJECT_DIR = "/content/Tumor-AI-training-model"
# Base YAML merged with hyperparameters below → runtime file (regenerated each run)
CONFIG_BASE = "configs/config_colab.yaml"
RUNTIME_CONFIG = "configs/config_colab_runtime.yaml"
EXTRA_ARGS = ""

# --- Hyperparameters (written to RUNTIME_CONFIG before training) ---
# ViT: try lr 1e-4 or 5e-5, EPOCHS 30–50, batch_size 32 if GPU allows (else 8–16).
# ResNet50: lr 3e-4 and batch_size 16–32 are strong defaults.
# If you change train/val/test splits, delete data/processed/ before re-running.
LR = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 8
MIN_DELTA = 0.001
SEED = 42
USE_CLASS_WEIGHTS = True
USE_WEIGHTED_SAMPLER = True
SAVE_LAST_CHECKPOINT = True

BATCH_SIZE = 16
NUM_WORKERS = 2
IMG_SIZE = 224
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

# --- Data source ---
DATA_SOURCE = "kaggle"

DRIVE_DATA_PATH = "/content/drive/MyDrive/brain_tumor_data"
DRIVE_USE_IN_PLACE = False

KAGGLE_JSON_DRIVE = "/content/drive/MyDrive/kaggle.json"
# =============================================================================

import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import yaml


def deep_merge(base, over):
    for k, v in over.items():
        if k in base and isinstance(base.get(k), dict) and isinstance(v, dict):
            deep_merge(base[k], v)
        else:
            base[k] = v
    return base


def run(cmd, cwd=None):
    print("+", cmd if isinstance(cmd, str) else " ".join(cmd))
    subprocess.check_call(cmd, cwd=cwd)


from google.colab import drive

drive.mount("/content/drive")
subprocess.run(["nvidia-smi"], check=False)

if Path(PROJECT_DIR).exists():
    print("Updating repo...")
    run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH])
    run(["git", "-C", PROJECT_DIR, "checkout", BRANCH])
    run(["git", "-C", PROJECT_DIR, "pull", "origin", BRANCH])
else:
    run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, PROJECT_DIR])

os.chdir(PROJECT_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
if DATA_SOURCE == "kaggle":
    run([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
print("Project:", PROJECT_DIR)

base_cfg_path = Path(PROJECT_DIR) / CONFIG_BASE
with open(base_cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

hyper_overrides = {
    "data": {
        "batch_size": int(BATCH_SIZE),
        "num_workers": int(NUM_WORKERS),
        "img_size": int(IMG_SIZE),
        "train_split": float(TRAIN_SPLIT),
        "val_split": float(VAL_SPLIT),
        "test_split": float(TEST_SPLIT),
    },
    "training": {
        "epochs": int(EPOCHS),
        "lr": float(LR),
        "weight_decay": float(WEIGHT_DECAY),
        "patience": int(PATIENCE),
        "min_delta": float(MIN_DELTA),
        "seed": int(SEED),
        "use_class_weights": bool(USE_CLASS_WEIGHTS),
        "use_weighted_sampler": bool(USE_WEIGHTED_SAMPLER),
        "save_last_checkpoint": bool(SAVE_LAST_CHECKPOINT),
    },
}
deep_merge(cfg, hyper_overrides)

runtime_path = Path(PROJECT_DIR) / RUNTIME_CONFIG
runtime_path.parent.mkdir(parents=True, exist_ok=True)
with open(runtime_path, "w", encoding="utf-8") as f:
    f.write("# Generated by Colab notebook cell; overwritten each run.\n")
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False, allow_unicode=True)
print("Wrote runtime config:", runtime_path)

data_dir_for_main = None

if DATA_SOURCE == "kaggle":
    raw = Path(PROJECT_DIR) / "data" / "raw"
    raw.mkdir(parents=True, exist_ok=True)

    kdir = Path.home() / ".kaggle"
    kdir.mkdir(parents=True, exist_ok=True)
    kaggle_dest = kdir / "kaggle.json"
    got = False
    if KAGGLE_JSON_DRIVE:
        src_k = Path(KAGGLE_JSON_DRIVE)
        if src_k.is_file():
            shutil.copy2(src_k, kaggle_dest)
            os.chmod(kaggle_dest, 0o600)
            print("Using kaggle.json from Drive:", src_k)
            got = True
    if not got:
        from google.colab import files

        print("Upload kaggle.json (Kaggle → Account → API → Create Token)")
        uploaded = files.upload()
        for name, data in uploaded.items():
            if name.endswith(".json"):
                kaggle_dest.write_bytes(data)
                os.chmod(kaggle_dest, 0o600)
                got = True
                break
        if not got:
            raise RuntimeError("Upload kaggle.json")

    data_parent = Path(PROJECT_DIR) / "data"
    zip_path = data_parent / "brain-tumor-mri-dataset.zip"
    subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            "-d",
            "masoudnickparvar/brain-tumor-mri-dataset",
            "-p",
            str(data_parent),
        ],
        check=True,
        cwd=PROJECT_DIR,
    )
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(raw)
    zip_path.unlink(missing_ok=True)

elif DATA_SOURCE == "drive":
    drive_root = Path(DRIVE_DATA_PATH).expanduser().resolve()
    if not drive_root.is_dir():
        raise FileNotFoundError(
            f"Not a directory: {drive_root}\n"
            "Use the path after mounting Drive, e.g. /content/drive/MyDrive/... or "
            "/content/drive/Shareddrives/<Name>/..."
        )

    if DRIVE_USE_IN_PLACE:
        subs_d = [
            p
            for p in drive_root.iterdir()
            if p.is_dir() and p.name not in {".ipynb_checkpoints"}
        ]
        if (
            len(subs_d) == 1
            and (subs_d[0] / "Training").is_dir()
            and not (drive_root / "Training").is_dir()
        ):
            effective_raw = subs_d[0]
            print("Using nested folder as raw data root:", effective_raw)
        else:
            effective_raw = drive_root
        if not (effective_raw / "Training").is_dir():
            raise FileNotFoundError(
                f"Expected a Training/ folder under {effective_raw}. "
                "Point DRIVE_DATA_PATH at the folder that contains Training/ (or one level above if a single nested folder holds it)."
            )
        data_dir_for_main = str(effective_raw)
        raw = effective_raw
        print("Drive in-place raw dir (--data_dir):", data_dir_for_main)
    else:
        raw = Path(PROJECT_DIR) / "data" / "raw"
        raw.mkdir(parents=True, exist_ok=True)
        for item in drive_root.iterdir():
            dest = raw / item.name
            if dest.exists():
                if dest.is_dir():
                    shutil.rmtree(dest)
                else:
                    dest.unlink()
            if item.is_dir():
                shutil.copytree(item, dest)
            else:
                shutil.copy2(item, dest)
else:
    raise ValueError('DATA_SOURCE must be "kaggle" or "drive"')

if DATA_SOURCE == "kaggle" or (DATA_SOURCE == "drive" and not DRIVE_USE_IN_PLACE):
    subs = [p for p in raw.iterdir() if p.is_dir() and p.name not in {".ipynb_checkpoints"}]
    if len(subs) == 1 and (subs[0] / "Training").is_dir():
        inner = subs[0]
        for item in inner.iterdir():
            shutil.move(str(item), str(raw / item.name))
        inner.rmdir()
        print("Flattened:", raw)

if (raw / "Training").is_dir():
    print("OK: Training/ found at", raw / "Training")
else:
    print("WARNING: expected Training/ under", raw)

import shlex

model_list = MODELS.split()
cmd = [
    sys.executable,
    "main.py",
    "--config",
    RUNTIME_CONFIG,
    "--models",
    *model_list,
    "--epochs",
    str(int(EPOCHS)),
]
if data_dir_for_main:
    cmd += ["--data_dir", data_dir_for_main]
if EXTRA_ARGS.strip():
    cmd += shlex.split(EXTRA_ARGS.strip())

os.chdir(PROJECT_DIR)
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=PROJECT_DIR)

results_dir = Path(PROJECT_DIR) / "results"
csv_path = results_dir / "final_results.csv"
if csv_path.is_file():
    print("\n--- final_results.csv ---\n")
    print(csv_path.read_text())
print("\nDone. Artifacts: results/checkpoints/, results/plots/, results/final_results.csv")

